In [1]:
from graphs import *
from RQAOA import *
import math

In [2]:
graph = generate_d_regular_graph(128, 3, seed=42, e_dist=[50, 20])
edges, adj_mat = graph_to_array(graph)
max_ang_freq = get_max_frequency(edges, adj_mat)
max_freq = max_ang_freq/(2*np.pi)

In [3]:
qaoa = partial(gamma_cost, edges, adj_mat)

def qaoa_sqaure(gamma):
    return qaoa(gamma)**2

In [4]:
def subdivision_algorithm(q, N, M, tol=1e-8, max_iter=50):
    """
    Implements the subdivision algorithm and returns both sqrt(q_tilde)
    and the x-value at which q_tilde is attained.

    Parameters
    ----------
    q : callable
        A function q(x) you want to evaluate.
    N : int
        Parameter N from the algorithm.
    M : int
        Initial subdivision parameter (must be > 2N).
    tol : float
        Tolerance for stopping criterion.
    max_iter : int
        Maximum number of refinement iterations.

    Returns
    -------
    (float, float)
        sqrt_q_tilde, x_tilde
        Where q_tilde is the maximum q-value found at iteration end (or break)
        and x_tilde is the location of that maximum.
    """
    # --- Initialize h and intervals ---
    h = np.pi / M
    # Each interval has width 2h, so the endpoints go 0, 2h, 4h, ..., 2π.
    # We'll store intervals as tuples (start, end).
    intervals = []
    num_intervals = M  # Because M * 2h = 2π
    for i in range(num_intervals):
        start = 2 * h * i
        end   = start + 2 * h
        intervals.append((start, end))

    # Make sure the last interval extends to 2π exactly (minor floating-error fix).
    intervals[-1] = (intervals[-1][0], 2*np.pi)

    iteration = 0
    x_tilde = None  # will store the location of max q-value
    q_tilde = None  # will store the max q-value

    while iteration < max_iter:
        iteration += 1

        # 1. Evaluate q at midpoints of each interval
        midpoints = [(a + b) / 2 for (a, b) in intervals]
        q_values = [q(t) for t in midpoints]

        # 2. Find q_tilde = max of q(t_i)
        i_max = np.argmax(q_values)
        q_tilde = q_values[i_max]
        x_tilde = midpoints[i_max]

        # 3. Check stopping criterion: q_tilde*(sec(N*h)-1) < tol
        secNh = 1.0 / np.cos(N * h)  # sec(x) = 1/cos(x)
        if q_tilde * (secNh - 1.0) < tol:
            print(f"Stopping at iteration {iteration}: condition is satisfied.")
            break

        # 4. Reject intervals with q(t_i) < q_tilde*cos(N*h)
        threshold = q_tilde * (np.cos(((N+4) * h)/2)**2)
        kept_intervals = []
        for (iv, val) in zip(intervals, q_values):
            if val >= threshold:
                kept_intervals.append(iv)

        # 5. Subdivide remaining intervals (reduce h accordingly).
        new_intervals = []
        for (a, b) in kept_intervals:
            mid = 0.5 * (a + b)
            new_intervals.append((a, mid))
            new_intervals.append((mid, b))

        intervals = new_intervals
        h /= 2.0  # halving the interval width

    # Return sqrt(q_tilde) and the location x_tilde
    return np.sqrt(q_tilde), x_tilde

In [5]:
opt_val, opt_loc = subdivision_algorithm(qaoa_sqaure, max_freq, math.ceil(np.pi*(2*max_freq + 1)), tol=1e-8, max_iter=50)

Stopping at iteration 25: condition is satisfied.


In [6]:
opt_val

3764.537706062272

In [7]:
opt_loc

3.135822669251456

In [8]:
n_samps = np.pi*(2*max_freq + 1)

In [9]:
opt_gamma = optimize.brute(qaoa, ((0, pi),), Ns=math.ceil(n_samps), workers=1)

In [10]:
opt_gamma

array([0.00577024])

In [11]:
qaoa(opt_gamma)

-3764.537698245403

In [12]:
qaoa(opt_loc)

-3764.537706062272